# Notebook 05: Policy Evaluation and Business Impact

The core business question is not only which model has the best curve, but which targeting policy creates the most incremental value under budget constraints.

## Offline policy assumptions (explicit)
- model ranking is based on predicted uplift
- incremental conversions are estimated from observed treatment/control outcomes within ranked buckets
- incremental revenue is approximated using average order value or observed spend assumptions
- this is offline policy evaluation, not causal proof of production revenue lift

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

if Path.cwd().name == 'notebooks':
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import set_seed, ensure_output_dirs, save_plot, format_currency
from src.preprocessing import load_hillstrom_data, basic_cleaning, split_features_outcomes_treatment, create_train_valid_test_split
from src.baselines import fit_naive_treated_response_model, predict_naive_treated_response_model, fit_two_model_uplift, predict_two_model_uplift
from src.meta_learners import XLearner
from src.modern_causal import DRLearnerWrapper
from src.evaluation import qini_curve, qini_coefficient, auuc_score, budget_sensitivity_table
from src.policy import compare_targeting_policies

SEED = 42
set_seed(SEED)
OUT_DIRS = ensure_output_dirs(PROJECT_ROOT)
DATA_PATH = PROJECT_ROOT / 'data' / 'hillstrom.csv'

In [ ]:
df = basic_cleaning(load_hillstrom_data(str(DATA_PATH)))
X, y, t = split_features_outcomes_treatment(df, outcome_col='conversion', treatment_col='treatment')
splits = create_train_valid_test_split(X, y, t, test_size=0.2, valid_size=0.2, random_state=SEED, stratify=True)
print('Shared split sizes:', {k: len(v) for k, v in splits.items() if k.startswith('X_')})

## 1) Train policy candidate models

In [ ]:
naive_model = fit_naive_treated_response_model(splits['X_train'], splits['y_train'], splits['t_train'], model_family='xgboost', random_state=SEED)
naive_scores = predict_naive_treated_response_model(naive_model, splits['X_test'])

two_t, two_c = fit_two_model_uplift(splits['X_train'], splits['y_train'], splits['t_train'], model_family='xgboost', random_state=SEED)
two_scores = predict_two_model_uplift(two_t, two_c, splits['X_test'])

x_model = XLearner(random_state=SEED).fit(splits['X_train'], splits['t_train'], splits['y_train'])
x_scores = x_model.predict_uplift(splits['X_test'])

dr_model = DRLearnerWrapper(random_state=SEED, prefer_econml=True).fit(splits['X_train'], splits['t_train'], splits['y_train'])
dr_scores = dr_model.predict_uplift(splits['X_test'])

score_dict = {
    'naive_response': naive_scores,
    'two_model_uplift': two_scores,
    'x_learner': x_scores,
    'dr_learner': dr_scores,
}
print('DR backend:', dr_model.backend_)

## 2) Side-by-side ranking metrics

In [ ]:
metric_rows = []
for name, score in score_dict.items():
    metric_rows.append({
        'model': name,
        'qini': qini_coefficient(splits['y_test'], splits['t_test'], score),
        'auuc': auuc_score(splits['y_test'], splits['t_test'], score),
    })
metric_df = pd.DataFrame(metric_rows).sort_values('qini', ascending=False).reset_index(drop=True)
display(metric_df)
metric_df.to_csv(OUT_DIRS['tables'] / 'nb05_model_metric_comparison.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for name, score in score_dict.items():
    c = qini_curve(splits['y_test'], splits['t_test'], score)
    ax.plot(c['fraction_targeted'], c['incremental_outcomes'], label=name)
ax.set_title('Uplift/Qini Curves Across Major Models')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative incremental conversions')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb05_model_qini_comparison.png')
plt.show()

## 3) Budget-constrained policy simulation

In [ ]:
# Average order value assumption from observed purchasers in full dataset.
positive_spend = df.loc[df['conversion'] == 1, 'spend']
AOV = float(positive_spend.mean()) if len(positive_spend) else 100.0
print('Average order value assumption:', format_currency(AOV))

In [ ]:
budgets = [i / 10 for i in range(1, 11)]
policy_rows = []
for model_name, score in score_dict.items():
    table = budget_sensitivity_table(
        splits['y_test'], splits['t_test'], score,
        budgets=budgets, average_order_value=AOV
    )
    table['model'] = model_name
    policy_rows.append(table)
policy_table = pd.concat(policy_rows, ignore_index=True)
display(policy_table.head(20))
policy_table.to_csv(OUT_DIRS['tables'] / 'nb05_budget_sensitivity_models.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for name, grp in policy_table.groupby('model'):
    ax.plot(grp['budget'], grp['incremental_revenue'], marker='o', label=name)
ax.set_title('Incremental Revenue vs Budget')
ax.set_xlabel('Targeting budget (fraction)')
ax.set_ylabel('Estimated incremental revenue (offline simulation)')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb05_budget_sensitivity_revenue.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for name, grp in policy_table.groupby('model'):
    ax.plot(grp['budget'], grp['incremental_conversions'], marker='o', label=name)
ax.set_title('Incremental Conversions vs Budget')
ax.set_xlabel('Targeting budget (fraction)')
ax.set_ylabel('Estimated incremental conversions')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb05_budget_sensitivity_conversions.png')
plt.show()

## 4) Compare policy candidates at a fixed budget (30%)

In [ ]:
budget = 0.30
policy_compare = compare_targeting_policies(
    splits['y_test'], splits['t_test'], budget=budget,
    uplift_score=dr_scores,
    response_score=naive_scores,
    average_order_value=AOV,
    random_state=SEED,
)
display(policy_compare)
policy_compare.to_csv(OUT_DIRS['tables'] / 'nb05_policy_compare_budget_30pct.csv', index=False)

## Recommendation
Deployment recommendation should balance:
- expected incremental value
- model complexity and maintainability
- operational latency and retraining cost

Choose the model with strongest budget-relevant policy lift, unless complexity costs dominate.

## Risk and caveats
- Offline uplift estimates are noisy and assumption-sensitive.
- ITE estimates are uncertain at individual level.
- Revenue mapping depends on AOV/proxy assumptions.
- External validity can drift across campaigns and seasons.
- Online experimentation is required before production rollout.

## Executive final summary
- Technical: policy-relevant model ranking differences are quantified via Qini/AUUC and budget curves.
- Business: model scores are translated into incremental conversion/revenue estimates under budget constraints.
- Next validation: online A/B or bandit-style rollout to verify production lift.